# annstat — Colab notebook: CIFAR-10 experiment (E9), significance tests (E10) and a quick audit

**Before running:** *Runtime → Change runtime type → GPU* (T4 is enough). Then *Runtime → Run all*.

What this notebook does
1. installs the package from GitHub and runs its test-suite,
2. runs a one-call statistical audit on Iris (sanity check),
3. trains VGG-style CNNs on **CIFAR-10** (PyTorch), builds the hidden-layer reductions (LSC-QR, OBS, ThiNet, neuron merging, CUR, random+LSC, magnitude) and audits the 256-unit layer (effective rank, ablation tests),
4. runs the formal significance tests across data sets/models (Friedman + Nemenyi + Wilcoxon-Holm),
5. generates the LaTeX tables and downloads a zip with every result.

Typical time on a T4: about 3-5 min per seed for 50k images and 20 epochs.

In [ ]:
# ---- configuration (edit if the repository is moved) ----
REPO_URL = "https://github.com/yamenetoo/Artificial-Neural-Network-Analysis-Using-Statistical-Approaches.git"
BRANCH   = "annstat-package"      # change to "main" after you merge / move the package
N_TRAIN, EPOCHS, SEEDS = 50000, 20, 5   # CIFAR-10 experiment size (use 20000, 12, 3 on CPU)
RUN_OLD_EXPERIMENTS = False       # E6/E7/E8 (tabular + MNIST baselines) are stored in experiments/results; set True to re-run them (~15 min)

In [ ]:
!git clone -q -b {BRANCH} {REPO_URL} annstat_repo
%cd annstat_repo
!pip -q install -e . tabulate
import annstat, torch
print("annstat", annstat.__version__, "| torch", torch.__version__, "| GPU:", torch.cuda.is_available())

In [ ]:
!python -m pytest -q tests 2>&1 | tail -3

## 1. Quick audit (Iris)

In [ ]:
!python examples/quickstart_iris.py | head -60

## 2. Audit your own PyTorch model (template)
`from_torch(model, hidden_module, head_module)` reads the activations of the last hidden layer through a forward hook.

In [ ]:
# Template - uncomment and adapt
# import numpy as np, annstat as A
# adapter = A.from_torch(model, hidden_module=model[15], head_module=model[16], classes=np.arange(10))
# rep = A.StatisticalTuner(adapter, seed=0).analyze(X_fit, y_fit, X_eval, y_eval, n_boot=500, input_tests_on=False,
#                                                    reduction_ks=[32, 64, 128], reduction_methods=("lsc_qr", "obs", "thinet"))
# print(rep.to_markdown())

## 3. E9 - CIFAR-10 (downloads the dataset with torchvision)

In [ ]:
import os
os.environ["ANNSTAT_DATA"] = "/content/data"
!cd experiments && python e9_cifar.py {N_TRAIN} {EPOCHS} {SEEDS}

In [ ]:
if RUN_OLD_EXPERIMENTS:
    !pip -q install mlxtend
    !cd experiments && python e6_baselines.py && python e7_torch_deep.py && python e8_bound.py
else:
    print("using the stored E6/E7/E8 results in experiments/results")

## 4. E10 - formal significance tests and LaTeX tables

In [ ]:
!cd experiments && python make_tables2.py && python e10_significance.py

In [ ]:
import pandas as pd
R = "experiments/results"
print(open(f"{R}/t_e9.tex").read())
print(open(f"{R}/t_e9_audit.tex").read())
print(open(f"{R}/t_e10_ranks.tex").read())
print(open(f"{R}/t_e10_pairs.tex").read())
display(pd.read_csv(f"{R}/e9_cifar_audit.csv"))
display(pd.read_csv(f"{R}/e10_pairs.csv").round(3))

## 5. Download everything
Send `colab_results.zip` back (or just the `e9_*`, `e10_*`, `t_e9*`, `t_e10*` files) to update the manuscript.

In [ ]:
!cd experiments && zip -rq ../colab_results.zip results
from google.colab import files
files.download("colab_results.zip")